## Training For Nature Communications

In [1]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from matplotlib import pyplot as plt
from sklearn.metrics import r2_score
import keras_nlp
from keras import regularizers
from keras.callbacks import History 
history = History()
np.set_printoptions(precision=4, suppress=True)
tf.config.set_visible_devices([], 'GPU') # Disable GPU Usage
print("TensorFlow version:", tf.__version__)

ValueError: Unable to import `keras` with `tensorflow`.  Please check your Keras and Tensorflow version are compatible; Keras 3 requires TensorFlow 2.15 or later. See keras.io/getting_started for more information on installing Keras.

In [ ]:
# Load data
arr = np.loadtxt("./training.csv", delimiter=",")
x = arr[:,0:8]
y = arr[:,8:]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print('Training Input Shape: ', x_train.shape)
print('Training Output Shape: ', y_train.shape)
print('Testing Input Shape: ', x_test.shape)
print('Testing Output Shape: ', y_test.shape)

In [ ]:
# Flipping the training samples that have \Delta y > 0
training_size = len(x_train)
x_train_processed = x_train.copy()
y_train_processed = y_train.copy()

for i in range(training_size):
    if x_train[i, 7] > 0:
        x_train_processed[i, 1] = -1 * x_train[i, 1] # Flip obstacle y-coordinates
        x_train_processed[i, 5] = -1 * x_train[i, 5] # Flip receiver y-coordinates
        x_train_processed[i, 7] = -1 * x_train[i, 7] # Flip \Delta y
        y_train_processed[i, 0] = -1 * y_train[i, 0] # Flip curving parameter
        y_train_processed[i, 2] = -1 * y_train[i, 2] # Flip steering adjustment

# Normalization of training data (recording the mean and standard deviation)
xmean = x_train_processed.mean(0)
xstd = x_train_processed.std(0)
x_train_processed -= xmean
x_train_processed /= xstd
ymean = y_train_processed.mean(0)
ystd = y_train_processed.std(0)
y_train_processed -= ymean
y_train_processed /= ystd

In [ ]:
# Similarly, preprocess the test samples
testing_size = len(x_test)
x_test_processed = x_test.copy()
y_test_processed = y_test.copy()

for i in range(testing_size):
    if x_test[i, 7] > 0:
        x_test_processed[i, 1] = -1 * x_test_processed[i, 1] # Flip obstacle y-coordinates
        x_test_processed[i, 5] = -1 * x_test_processed[i, 5] # Flip receiver y-coordinates
        x_test_processed[i, 7] = -1 * x_test_processed[i, 7] # Flip \Delta y
        y_test_processed[i, 0] = -1 * y_test_processed[i, 0] # Flip curving parameter
        y_test_processed[i, 2] = -1 * y_test_processed[i, 2] # Flip steering adjustment

# Normalization of testing data (using mean and std learned from the training set)
x_test_processed -= xmean
x_test_processed /= xstd
y_test_processed -= ymean
y_test_processed /= ystd

In [ ]:
# Linear best model

model = tf.keras.models.Sequential([
  # tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=regularizers.L1(0.0001)),
  tf.keras.layers.Dense(512, activation='relu'),
  tf.keras.layers.Dropout(0.15),
  tf.keras.layers.Dense(512, activation='relu'),
  tf.keras.layers.Dropout(0.15),
  tf.keras.layers.Dense(512, activation='relu'),
  tf.keras.layers.Dropout(0.15),
  tf.keras.layers.Dense(3)
])
model.compile(optimizer=tf.keras.optimizers.legacy.SGD(learning_rate=0.008, momentum=0.8),
              loss=tf.keras.losses.MeanAbsoluteError())
history = model.fit(x_train_processed, y_train_processed, batch_size=64, epochs=1000, validation_data=(x_test_processed, y_test_processed))

25/25 [==============================] - 0s 5ms/step - loss: 0.3903 - val_loss: 0.3803
Epoch 163/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3924 - val_loss: 0.3794
Epoch 164/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3927 - val_loss: 0.3803
Epoch 165/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3885 - val_loss: 0.3793
Epoch 166/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3860 - val_loss: 0.3786
Epoch 167/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3909 - val_loss: 0.3804
Epoch 168/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3840 - val_loss: 0.3791
Epoch 169/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3873 - val_loss: 0.3799
Epoch 170/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3845 - val_loss: 0.3785
Epoch 171/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3855 - val_loss: 0.

25/25 [==============================] - 0s 5ms/step - loss: 0.3502 - val_loss: 0.3559
Epoch 323/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3468 - val_loss: 0.3544
Epoch 324/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3447 - val_loss: 0.3557
Epoch 325/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3425 - val_loss: 0.3550
Epoch 326/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3457 - val_loss: 0.3578
Epoch 327/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3435 - val_loss: 0.3544
Epoch 328/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3462 - val_loss: 0.3549
Epoch 329/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3450 - val_loss: 0.3541
Epoch 330/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3422 - val_loss: 0.3551
Epoch 331/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3488 - val_loss: 0.

25/25 [==============================] - 0s 5ms/step - loss: 0.3229 - val_loss: 0.3463
Epoch 483/1000
25/25 [==============================] - 0s 7ms/step - loss: 0.3150 - val_loss: 0.3459
Epoch 484/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.3184 - val_loss: 0.3464
Epoch 485/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.3208 - val_loss: 0.3473
Epoch 486/1000
25/25 [==============================] - 0s 8ms/step - loss: 0.3185 - val_loss: 0.3446
Epoch 487/1000
25/25 [==============================] - 0s 7ms/step - loss: 0.3154 - val_loss: 0.3462
Epoch 488/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.3182 - val_loss: 0.3460
Epoch 489/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.3197 - val_loss: 0.3484
Epoch 490/1000
25/25 [==============================] - 0s 7ms/step - loss: 0.3138 - val_loss: 0.3457
Epoch 491/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.3208 - val_loss: 0.

25/25 [==============================] - 0s 5ms/step - loss: 0.2978 - val_loss: 0.3471
Epoch 643/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2993 - val_loss: 0.3461
Epoch 644/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2983 - val_loss: 0.3468
Epoch 645/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2993 - val_loss: 0.3475
Epoch 646/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.3022 - val_loss: 0.3458
Epoch 647/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2948 - val_loss: 0.3504
Epoch 648/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2963 - val_loss: 0.3509
Epoch 649/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2968 - val_loss: 0.3510
Epoch 650/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2940 - val_loss: 0.3489
Epoch 651/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2966 - val_loss: 0.

25/25 [==============================] - 0s 5ms/step - loss: 0.2787 - val_loss: 0.3523
Epoch 803/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2813 - val_loss: 0.3493
Epoch 804/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2794 - val_loss: 0.3473
Epoch 805/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2829 - val_loss: 0.3484
Epoch 806/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2816 - val_loss: 0.3504
Epoch 807/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2789 - val_loss: 0.3522
Epoch 808/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2820 - val_loss: 0.3500
Epoch 809/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2782 - val_loss: 0.3540
Epoch 810/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2816 - val_loss: 0.3503
Epoch 811/1000
25/25 [==============================] - 0s 6ms/step - loss: 0.2834 - val_loss: 0.

25/25 [==============================] - 0s 5ms/step - loss: 0.2616 - val_loss: 0.3504
Epoch 963/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2672 - val_loss: 0.3507
Epoch 964/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2616 - val_loss: 0.3533
Epoch 965/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2650 - val_loss: 0.3556
Epoch 966/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2670 - val_loss: 0.3498
Epoch 967/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2650 - val_loss: 0.3528
Epoch 968/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2641 - val_loss: 0.3525
Epoch 969/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2651 - val_loss: 0.3518
Epoch 970/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2623 - val_loss: 0.3511
Epoch 971/1000
25/25 [==============================] - 0s 5ms/step - loss: 0.2699 - val_loss: 0.

In [ ]:
# model.save('physicalBest.keras')

model = tf.keras.models.load_model('modelParam.keras')
model.summary()

# np.savetxt("y_pred1.csv", y_pred, delimiter=",")
# np.savetxt("y_test1.csv", y_test, delimiter=",")
# np.savetxt("x_test1.csv", x_test, delimiter=",")

In [ ]:
y_pred = model.predict(x_test_processed)
y_pred_tr = model.predict(x_train_processed)

In [ ]:
# Unnormalize data
y_pred = y_pred * ystd
y_pred = y_pred + ymean
y_pred_tr = y_pred_tr * ystd
y_pred_tr = y_pred_tr + ymean

for i in range(testing_size):
    if x_test[i, 7] > 0:
        y_pred[i, 0] = -1 * y_pred[i, 0] # Flip curving parameter
        y_pred[i, 2] = -1 * y_pred[i, 2] # Flip steering adjustment

for i in range(training_size):
    if x_train[i, 7] > 0:
        y_pred_tr[i, 0] = -1 * y_pred_tr[i, 0] # Flip curving parameter
        y_pred_tr[i, 2] = -1 * y_pred_tr[i, 2] # Flip steering adjustment

In [ ]:
ind = 0
plt.scatter(y_test[:,ind], y_pred[:,ind])
plt.axis('square')
plt.xlim([-10, 10])
plt.ylim([-10, 10])
print('R2 score: ', r2_score(y_test[:,ind],y_pred[:,ind]))
print('All R2: ', r2_score(y_test[:,0],y_pred[:,0]), r2_score(y_test[:,1],y_pred[:,1]), r2_score(y_test[:,2],y_pred[:,2]))
print('Total R2: ', r2_score(y_test[:,0],y_pred[:,0]) + r2_score(y_test[:,1],y_pred[:,1]) + r2_score(y_test[:,2],y_pred[:,2]))

In [ ]:
plt.scatter(y_train[:,0], y_pred_tr[:,0])
plt.axis('square')
plt.xlim([-10, 10])
plt.ylim([-10, 10])
print('R2 score: ',r2_score(y_train[:,0],y_pred_tr[:,0]))

In [ ]:
plt.hist(y_pred[:,0], bins=20, range=(-10,10))

In [ ]:
plt.hist(y_test[:,0], bins=20, range=(-10,10))

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss')
plt.legend()

## For Manual Testing of Neural Network

In [ ]:
model = tf.keras.models.load_model('modelParam.keras')
model.summary()

In [ ]:
# For manual testing individual cases
manual_test = np.array([0.15, -0.06, 0.02, 0.08, 0.3, -0.08, 0.74129, 0.04])
manual_test = np.reshape(manual_test, (1,8))

In [ ]:
manual_test_processed = manual_test.copy()
if manual_test[0][7] > 0:
        manual_test_processed[0][1] = -1 * manual_test_processed[0][1] # Flip obstacle y-coordinates
        manual_test_processed[0][5] = -1 * manual_test_processed[0][5] # Flip receiver y-coordinates
        manual_test_processed[0][7] = -1 * manual_test_processed[0][7] # Flip \Delta y
manual_test_processed -= xmean
manual_test_processed /= xstd
manual_pred = model.predict(manual_test_processed)
manual_pred = manual_pred * ystd
manual_pred = manual_pred + ymean
if manual_test[0][7] > 0:
        manual_pred[0][0] = -1 * manual_pred[0][0] # Flip curving parameter
        manual_pred[0][2] = -1 * manual_pred[0][2] # Flip steering adjustment

In [ ]:
manual_pred

In [ ]:
pip install tensorflow==2.15.0